# Toy Data Set
Mock Data values loaded into a panda df for this mock training pipeline

In [ ]:
import pandas as pd
from datasets import Dataset

mars_toy_data = {
    "hypothesis": [
        "Why does the fourth planet from the sun appear to have a reddish hue?",
        "What geological composition is responsible for the distinct color of Mars?",
        "Could you explain the reason behind the Martian surface looking like rust?",
        "I am looking at Mars through a telescope; what causes that red tint?"
    ],
    "conclusion": [
        "Mars appears red because its surface is covered in iron oxide, commonly known as rust.",
        "The distinct reddish color of the planet is a direct result of abundant iron oxide in its regolith.",
        "The rusty appearance is caused by the oxidation of iron-rich minerals across the Martian landscape.",
        "That red tint is due to a thick layer of iron oxide dust covering the planet's surface."
    ]
}
df_mars = pd.DataFrame(mars_toy_data)

df_mars = df_mars.rename(columns={"hypothesis": "anchor", "conclusion": "positive"})

dataset = Dataset.from_pandas(df_mars)

# Fine Tunning bge-m3

This code block is for fine tunning the semantic model, bge-m3. The semantic model is responsible for embedding our hypotheis and conclusions in some meaningful way. Training the model to from a pull hypothesis-conclusion pairs together in the embedding space. This will then allow us to steer the LLM.

In [ ]:
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, losses
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

# 1. Load BGE-M3 (or bge-small-en-v1.5 for ultra-fast local prototyping)
embedding_model = SentenceTransformer("BAAI/bge-m3")

# 2. Define the contrastive loss function
loss = losses.MultipleNegativesRankingLoss(embedding_model)

# 3. Setup lightweight training args
args = SentenceTransformerTrainingArguments(
    output_dir="./bge_m3_toy_checkpoint",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    learning_rate=2e-5,
    logging_steps=1,
    save_strategy="no"
)

trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=dataset,
    loss=loss
)

trainer.train()

# Calculating the relationship
Here the average directional vector between our conclusions and hypothesis are measured across the data set. This is used to create one driectional vector encapsulating this relation ship. _Vector of truth for the real ones who know_.

In [ ]:
import torch

# Encode the training samples
hyp_embeddings = embedding_model.encode(mars_toy_data["hypothesis"], convert_to_tensor=True)
con_embeddings = embedding_model.encode(mars_toy_data["conclusion"], convert_to_tensor=True)

# Calculate mean translation vector Δv
delta_v = (con_embeddings - hyp_embeddings).mean(dim=0)
delta_v = delta_v / torch.norm(delta_v)  # Normalize
print(f"Computed steering vector norm: {torch.norm(delta_v):.4f}, Shape: {delta_v.shape}")

# Activation Steering via PyTorch Hooks
This section demonstrates how to intervene directly in the generative model's forward pass. Because the embedding model (BGE-M3) and the language model (Qwen2.5) operate in different latent dimensional spaces, we must first project the steering vector to match the LLM's hidden size. We then use a PyTorch forward hook to forcefully inject this modified vector into the residual stream at a specific transformer layer.

Component Documentation

`Dimensionality Projection`: Maps the 1024-dimensional space of the embedding vector to the 896-dimensional hidden state of the 0.5B model using a linear transformation without biases.

`Forward Hook` (steering_hook): Intercepts the matrix computation at target_layer. It selectively adds the projected vector (scaled by steering_strength) to the hidden state of the last generated token ([:, -1, :]) before passing it up to the next layer.

`Hook Management`: The hook is registered before the steered generation phase and explicitly removed (hook_handle.remove()) immediately after to prevent corrupting future baseline inferences.

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
llm = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32)

# Project BGE-M3 embedding dimension (1024) to LLM hidden dimension (896 for Qwen2.5-0.5B)
embed_dim = embedding_model.get_sentence_embedding_dimension()  # 1024
llm_dim = llm.config.hidden_size                               # 896

projector = nn.Linear(embed_dim, llm_dim, bias=False)
projected_steering_vector = projector(delta_v).detach()

# Define activation hook
steering_strength = 1.5
target_layer = 10  # Apply to a middle layer

def steering_hook(module, input, output):
    # output is a tuple (hidden_states, ...)
    hidden_states = output[0]
    # Add steering vector to the last generated token's hidden state
    hidden_states[:, -1, :] += steering_strength * projected_steering_vector.to(hidden_states.device)
    return (hidden_states,) + output[1:]

prompt = "Methodology: We applied low-rank adaptation (LoRA) to train the model on domain text.\nConclusion:"
inputs = tokenizer(prompt, return_tensors="pt")

# 1. Unsteered generation baseline
unsteered_out = llm.generate(**inputs, max_new_tokens=40, do_sample=False)
print("--- Baseline Generation ---")
print(tokenizer.decode(unsteered_out[0], skip_special_tokens=True))

# 2. Steered generation
hook_handle = llm.model.layers[target_layer].register_forward_hook(steering_hook)
steered_out = llm.generate(**inputs, max_new_tokens=40, do_sample=False)
hook_handle.remove()

print("\n--- Steered Generation ---")
print(tokenizer.decode(steered_out[0], skip_special_tokens=True))